# Running the TSFM ensemble on the GIFT-Eval benchmark

**The following notebook is intended to reproduce GIFT-Eval ensemble results.**

Download the GIFT-Eval benchmark and set the `GIFT_EVAL` environment variable
before running this notebook. We use GIFT-Eval's `Dataset` class to load the data
and generate the evaluation windows. See the
[dataset notebook](https://github.com/SalesforceAIResearch/gift-eval/blob/main/notebooks/dataset.ipynb)
for an introduction to this interface.

We first run a small dataset for brevity. You can evaluate other datasets by
changing the `datasets` argument below, or omit it to run the full benchmark.

Use the same model sources, checkpoints, hardware, and library versions as the
reference experiment when reproducing results. Record the GPU model, CUDA driver,
PyTorch and Transformers versions with the final submission.


## TSFM installation

1. Clone the [GIFT-Eval repository](https://github.com/SalesforceAIResearch/gift-eval).
2. Follow its [installation and dataset instructions](https://github.com/SalesforceAIResearch/gift-eval#installation).
3. Obtain the Granite TSFM checkout containing these ensemble extensions.
   Its PatchTST implementation is used for both r1 and r2.
4. Follow the accompanying [README](README.md) to create the Python 3.12 benchmark
   environment and select it as the notebook kernel.

### Using the Granite TSFM source

Run this notebook from within the Granite checkout containing this runner.
Both PatchTST-r1 and r2 load directly from this checkout's `tsfm_public`
package. No separate PatchTST source directory is required.

These extensions must be available in the selected checkouts. Installing the
public Granite package alone does not establish the reference experiment setup.

## Imports

### Update the Python path and load the evaluation runner

The following cell selects the Granite source, loads environment variables, and
imports `run_evaluation()`. The source checkout is discovered from the working
directory and its parents. Restart the kernel
after switching source checkouts or editing experiment_config.json.


In [ ]:
import os
import sys
from pathlib import Path

granite_source = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "pyproject.toml").is_file() and (p / "tsfm_public").is_dir()),
    None,
)
if granite_source is None:
    raise RuntimeError("Run this notebook from within the Granite TSFM checkout")
runner_dir = granite_source / "notebooks/hfdemo/gift_eval_ensemble"
from dotenv import load_dotenv
load_dotenv()
for name in ("GIFT_EVAL",):
    if not os.environ.get(name) or not Path(os.environ[name]).expanduser().is_dir():
        raise RuntimeError(f"Set {name} to an existing directory")
sys.path[:0] = [str(granite_source), str(runner_dir)]
from run_gift_eval import EXPERIMENT_CONFIG, run_evaluation


## Load experiment settings

The runner loads [experiment_config.json](experiment_config.json) from this directory.
It defines the named member recipes, aggregation methods, quantile grid, dataset
lists/order, forecast terms, and execution defaults. Edit that file to change the
experiment and restart the kernel to reload it.

The default recipe is the research-and-Granite ensemble. To select Granite-only
members, uncomment the assignment below. PatchTST NaN filling defaults to enabled;
device selection is automatic (CUDA, then MPS, then CPU). For reference replication,
match the hardware and preprocessing settings used by the reference experiment.

The benchmark adapters emit quantiles 0.1 through 0.9. The runner computes the same
GIFT-Eval metrics for notebook and CLI runs.

In [ ]:
SETTINGS = EXPERIMENT_CONFIG["defaults"].copy()
# SETTINGS["model_name_config"] = "probability-ensemble-uniform-ibm-tsfm-granite-pt"
# SETTINGS["device"] = "cuda"  # Explicit device for reference replication.
SETTINGS["skip_processed"] = True

MODEL = SETTINGS["model_name_config"]
OUTPUT_ROOT = Path.cwd() / "replication_runs"
print(SETTINGS)

## Evaluation

Now that the model configuration and output paths are set, we will evaluate the
ensemble on the GIFT-Eval benchmark. The notebook and command-line script use
the same `run_evaluation()` function, which loads the models for each task and
runs them over the dataset's generated windows.

Following GIFT-Eval's naming convention, results are saved in `all_results.csv`
under `<out_dir>/<model_name_config>/`. The first column identifies each task
by dataset name, frequency, and horizon term:

```python
f"{dataset_name}/{freq}/{term}"
```

### Iterate through a small dataset

Start with `us_births/M` to check model loading and reproduce the monthly births
smoke result. This run has its own output directory. Change `datasets` to evaluate
a different subset; each selected dataset is evaluated for all eligible terms.

`skip_processed=True` resumes a run by skipping task names already present in
the CSV. Use a fresh output directory when changing the experiment settings.
Task failures are recorded in `execution_log.csv`; inspect that file and any
skipped-member warnings before accepting the results.


In [ ]:
smoke_settings = {
    **SETTINGS,
    "datasets": ["us_births/M"],
    "out_dir": OUTPUT_ROOT / "smoke",
}
smoke_csv = run_evaluation(**smoke_settings)

import pandas as pd
display(pd.read_csv(smoke_csv))

### Iterate through all datasets

Set `RUN_FULL_BENCHMARK=True` to evaluate the full dataset list and its eligible
short, medium, and long horizons. The full run writes to a separate directory.

If an interactive session is too short, run the same evaluation through the CLI
in a batch job and load the resulting CSV for inspection. See the accompanying
[README](README.md) for the command-line options.


In [ ]:
RUN_FULL_BENCHMARK = False
full_csv = OUTPUT_ROOT / "full" / MODEL / SETTINGS["out_name"]
if RUN_FULL_BENCHMARK:
    full_settings = {**SETTINGS, "datasets": None, "out_dir": OUTPUT_ROOT / "full"}
    full_csv = run_evaluation(**full_settings)
else:
    print("Full evaluation not started.")

### Finalize results and display

After the full evaluation completes, load `all_results.csv` and check that it
contains 97 unique task rows for the default full benchmark, the selected model name, and finite values for all
11 metric columns. These checks verify the row count and metric validity;
compare the results with the reference benchmark before submitting them.


In [ ]:
if full_csv.is_file():
    import numpy as np
    results = pd.read_csv(full_csv).sort_values("dataset")
    assert len(results) == results["dataset"].nunique() == 97
    assert results["model"].eq(MODEL).all()
    metrics = results.filter(regex=r"^eval_metrics/")
    assert metrics.shape[1] == 11 and np.isfinite(metrics.to_numpy()).all()
    display(results)
else:
    print("Full results are not available yet.")


## Submitting results

Follow the
[GIFT-Eval submission instructions](https://github.com/SalesforceAIResearch/gift-eval#submitting-your-results)
to contribute `all_results.csv` and the model's `config.json` under its results
directory. Link the public replication code and document the ensemble members,
source revisions, checkpoint revisions, and execution environment.
